In [ ]:
from pathlib import Path
import sys

src_dir = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "common.py").is_file() and (p / "feature_preference").is_dir())
sys.path.insert(0, str(src_dir))


In [ ]:
%load_ext autoreload
%autoreload 2
import os
import numpy as np
import random
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import cv2
import torch
import torchvision
from torchvision.transforms.functional import to_pil_image
import torchvision.transforms.v2 as Transform
from common import *
import seaborn as sns

In [ ]:
device = torch.device("cuda")
sns.set_theme(style="whitegrid", font="cmr10", font_scale=1.9)
plt.rcParams["axes.formatter.use_mathtext"] = True
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42})
colors = sns.color_palette()
colors

In [ ]:
ratios = [0.05, 0.075, 0.1, 0.125, 0.15]
mnist_dir = '/home/dataset/mnist'
background_predictability = 0.70
img_size = 48
min_mask_ratio = 0.10
colors = F.normalize(torch.tensor([
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, 1],
    [1, 1, 0],
    [1, 0, 1],
    [0, 1, 1],
    [2, 1, 0],
    [2, 0, 1],
    [1, 2, 0],
    [0, 1, 2]
], dtype=torch.float32)) * 255
colors

In [ ]:
bias = 0.05
def get_max_bounding_box(mask):
    # Find indices where mask is 1
    y_indices, x_indices = np.where(mask)
    # If no 1s are found, return None
    if len(y_indices) == 0:
        return (0, 0, 0, 0)

    # Get min and max coordinates
    min_y, max_y = np.min(y_indices), np.max(y_indices)
    min_x, max_x = np.min(x_indices), np.max(x_indices)

    return (min_y, min_x, max_y, max_x)

def get_ratio(mask):
    bbox = get_max_bounding_box(mask)
    min_y, min_x, max_y, max_x = bbox
    bbox_area = (max_y - min_y + 1) * (max_x - min_x + 1)
    if bbox_area < 10:
        return 0
    mask_area = torch.sum(mask)
    ratio = mask_area / bbox_area

    return ratio.item()

In [ ]:
mnist_train = torchvision.datasets.MNIST(root=mnist_dir, train=True, download=False, transform=Transform.Compose([Transform.ToTensor()]))
mnist_test = torchvision.datasets.MNIST(root=mnist_dir, train=False, download=False, transform=Transform.Compose([Transform.ToTensor()]))

total_pixel = img_size * img_size
mask_ratios = [[], []]
pure_pixel_ratios = [[], []]
filtered_img_ids = [[], []]
imgs = [[[] for _ in range(10)] for _ in range(2)]

print("calculate mask ratio...")
for i in range(2):
    for index, (x, label) in tqdm(enumerate([mnist_train, mnist_test][i])):
        ratio = get_ratio(x.squeeze() > bias)
        pure_pixel_ratios[i].append(torch.sum(x.squeeze() > bias) / (28*28))
        mask_ratios[i].append(ratio)
        if ratio < min_mask_ratio:
            filtered_img_ids[i].append(index)
            print(f"Filtering {index} with mask ratio {ratio:.2f}")
        else:
            imgs[i][label].append(x)
print(len(filtered_img_ids[0]), len(filtered_img_ids[1]))

In [ ]:
plt.plot(np.sort(mask_ratios[0]))
plt.plot(np.sort(mask_ratios[1]))
plt.axhline(min_mask_ratio, color='r', linestyle='--')
plt.title('mask ratio')
plt.show()
plt.plot(np.sort(pure_pixel_ratios[0]))
plt.plot(np.sort(pure_pixel_ratios[1]))
plt.title('pure pixel ratio')
plt.show()
print(min(pure_pixel_ratios[0]), max(pure_pixel_ratios[0]))
print(min(pure_pixel_ratios[1]), max(pure_pixel_ratios[1]))

In [ ]:
def colorize(image, label, target_pixel):
    (min_y, min_x, max_y, max_x) = get_max_bounding_box(image.squeeze())
    actual_data = image[:, min_y:max_y+1, min_x:max_x+1]
    scale = torch.sqrt(target_pixel / (actual_data > bias).sum()).item()
    new_height = min(img_size, round(scale * actual_data.shape[1]))
    new_width = min(img_size, round(scale * actual_data.shape[2]))
    scaled_img = torch.permute(actual_data, [1, 2, 0])
    scaled_img = cv2.resize(src=scaled_img.numpy(), dsize=(new_width, new_height))
    scaled_img = torch.asarray(scaled_img).unsqueeze(dim=0)
    x_offset = max(0, (img_size - new_width) // 2)
    y_offset = max(0, (img_size - new_height) // 2)
    background = torch.ones(3, img_size, img_size)
    result = torch.zeros(3, img_size, img_size)
    background = background * colors[label].unsqueeze(dim=-1).unsqueeze(dim=-1) / 255
    result[:, y_offset:y_offset + scaled_img.shape[1], x_offset:x_offset + scaled_img.shape[2]] = scaled_img
    # result = result * (result > bias).float() + background * (result <= bias).float()
    result = (1 - result) * result.float() + background * (1 - result)
    return 1 - result

In [ ]:
correct_label = [[round(len(imgs[t][i]) * background_predictability) for i in range(10)] for t in range(2)]
uncorrect_label = [[len(imgs[t][i]) - correct_label[t][i] for i in range(10)] for t in range(2)]
print(correct_label)
print(uncorrect_label)
for r in ratios:
    target_pixel = round(0.2 * total_pixel)
    converted_img = colorize(imgs[0][5][0], 2, target_pixel)
    plt.imshow(to_pil_image(converted_img))
    plt.show()
    break

In [ ]:
correct_label = [[round(len(imgs[t][i]) * background_predictability) for i in range(10)] for t in range(2)]
uncorrect_label = [[len(imgs[t][i]) - correct_label[t][i] for i in range(10)] for t in range(2)]
print(correct_label)
print(uncorrect_label)
for r in ratios:
    target_pixel = round(r * total_pixel)
    output_dir = f'/home/dataset/cmnist/{r}'
    counts = [[0 for j in range(10)] for i in range(2)]
    for t in range(2):
        set_seed(0)
        for label in range(10):
            indexes = torch.randperm(len(imgs[t][label]))
            correct_indexes = indexes[:correct_label[t][label]]
            uncorrect_indexes = indexes[correct_label[t][label]:]
            if t == 0:
                output_path = os.path.join(output_dir, f"train/{label}")
                os.makedirs(f"{output_dir}/train/{label}", exist_ok=True)
            else:
                output_path = os.path.join(output_dir, f"test/{label}")
                os.makedirs(f"{output_dir}/test/{label}", exist_ok=True)

            lists = [i for i in range(10)]
            lists.remove(label)

            for index in tqdm(correct_indexes):
                op = os.path.join(output_path, f"{counts[t][label]}.png")
                converted_img = colorize(imgs[t][label][index], label, target_pixel)
                to_pil_image(converted_img).save(op)
                counts[t][label] += 1
            for index in tqdm(uncorrect_indexes):
                op = os.path.join(output_path, f"{counts[t][label]}.png")
                converted_img = colorize(imgs[t][label][index], lists[random.randint(0, len(lists) - 1)], target_pixel)
                to_pil_image(converted_img).save(op)
                counts[t][label] += 1
    print(counts)